In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../")

In [3]:
from langevin_sampler import MetropolisAdjustedLangevinSampler, validate_sampler, visualise_rhat
import numpy as np
import jax
import jax.numpy as jnp

In [ ]:
# Sample from a high-dimensional standard normal distribution
mu = jnp.cos(jnp.linspace(0, 2 * jnp.pi, 100)) * 5
sigma2 = 0.2
reference_samples = jax.random.normal(jax.random.PRNGKey(0), (50, 100)) * jnp.sqrt(sigma2) + mu
print(reference_samples.shape)

In [ ]:
target_log_density = lambda x: -0.5 * jnp.sum((x - mu)**2) / sigma2
target_score_func = lambda x: -(x - mu) / sigma2
dim = 100
num_samples = 8_000

langevin_sampler = MetropolisAdjustedLangevinSampler(target_log_prob_fn=target_log_density,
                                                    target_score_fn=target_score_func,
                                                    shape=dim,
                                                    sample_from_simplex=False,
                                                    num_parallel_chains=10)

samples, num_accepted_samples, diagnostics, mh_ratio = langevin_sampler.sample(num_samples, with_diagnostics=True,)


In [ ]:
diagnostics

In [ ]:
samples.shape

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

plt.hist(mh_ratio[4]);

In [ ]:
validate_sampler(samples[:8000], mu, sigma2, reference_samples, 0.5)

In [ ]:
diagnostics.keys()

In [ ]:
rhat_mat = diagnostics["r_hat"].reshape((10, 10))
plt.figure(figsize=(8, 10))
im = plt.imshow(rhat_mat, cmap="YlOrRd", vmin=0.0, vmax=1.1)
plt.colorbar(im, label="R-hat")
plt.title("Convergence heatmap (R-hat)")
plt.show()


In [ ]:
# Caterpillar plot
import arviz_plots as az_plt
import arviz as az
import xarray as xr

rhat_mat = diagnostics["r_hat"].reshape((10, 10))
idata = az.from_dict(posterior={"transport_plan": np.asarray(samples[:100, :].reshape(-1, 10, 10))})
custom_rhat_ds = xr.Dataset({"r_hat": (["row", "col"], rhat_mat)})

az.plot_forest(idata, kind="forestplot", var_names=["transport_plan"], combined=True,)

In [ ]:
from langevin_sampler import plot_transport_plans_traces

plot_transport_plans_traces(samples, 10, 8000)

### Testing with the HFPDOT prior

In [4]:
from langevin_sampler import HFPDOTHyperprior
import jax
import ot
import ot.plot
import jax.numpy as jnp 
from sklearn.metrics.pairwise import pairwise_distances

In [5]:
# Support points
from datetime import date

init_key = jax.random.key(int(date.today().strftime("%Y%m%d")))
key, sub_key = jax.random.split(init_key)
p0_x = jax.random.truncated_normal(key,
                                lower = 1e-6,
                                upper = 1,
                                shape=(10,),) * 10

p1_x = jax.random.truncated_normal(sub_key,
                                lower = 1e-6,
                                upper = 1,
                                shape=(10,),) * 0.6

# Compute the Euclidean cost matrix
x = jnp.arange(10).reshape(-1, 1)
cost_matrix = pairwise_distances(x, x, metric="sqeuclidean", n_jobs=-1)
cost_matrix /= jnp.median(cost_matrix)


# Dimension of the support, very big...
dim = 100

target_prior = HFPDOTHyperprior(mu_0=p0_x,
                                nu_0=p1_x,
                                lambda_1=0.01,
                                lambda_2=0.01,
                                lambda_I_1=0.01,
                                lambda_I_2=0.01,
                                cost_fn=cost_matrix,
                                epsilon=1e-2)

target_log_density = target_prior.hyperprior_log_prob_fun
target_score_func = target_prior.hyperprior_score_fun

In [16]:
num_samples = 10_000

langevin_sampler = MetropolisAdjustedLangevinSampler(target_log_prob_fn=target_log_density,
                                                    target_score_fn=target_score_func,
                                                    shape=dim,
                                                    sample_from_simplex=False,
                                                    num_parallel_chains=10,
                                                    num_burnin=10_000)

In [17]:
samples, num_accepted_samples, diag, mh_log_ratio = langevin_sampler.sample(num_samples, with_diagnostics=True)

Initial states shape: (10, 100)
samples after flattening: (Array(80000, dtype=int32, weak_type=True), Array(100, dtype=int32, weak_type=True))


W0406 23:39:42.251104 1126824 spmd_partitioner.cc:663] [SPMD] Involuntary full rematerialization. The compiler cannot go from sharding {devices=[2]<=[2]} to {maximal device=0} efficiently for HLO operation %neg.9 = f32[40000]{0} negate(%add.791), sharding={devices=[2]<=[2]}, metadata={op_name="jit(compute_mcmc_diagnostics)/neg" stack_frame_id=80}. As the last resort, SPMD will replicate the tensor and then partition it to obtain the target sharding, which is inefficient. This issue will be fixed by Shardy partitioner in the future, which is tracked in b/433785288. Contact Shardy or XLA team for help.
W0406 23:39:42.251227 1126824 spmd_partitioner.cc:663] [SPMD] Involuntary full rematerialization. The compiler cannot go from sharding {devices=[2]<=[2]} to {maximal device=0} efficiently for HLO operation %sub.100 = f32[40000]{0} subtract(%copy, %copy), sharding={devices=[2]<=[2]}, metadata={op_name="jit(compute_mcmc_diagnostics)/jit(diff)/sub"}. As the last resort, SPMD will replicate th

Z-scores shape: (Array(10, dtype=int32), Array(8000, dtype=int32), Array(100, dtype=int32))
Shape of split chains: (Array(20, dtype=int32), Array(4000, dtype=int32), Array(100, dtype=int32))
var_plus: [1.0346661 1.0348386 1.034829  1.034824  1.0348207 1.034818  1.0348159
 1.0348141 1.0348127 1.0348114 1.0348101 1.0348092 1.0348082 1.0348072
 1.0348065 1.0348059 1.0348052 1.0348045 1.0348039 1.0348034 1.0348028
 1.0348022 1.0348017 1.0348014 1.034801  1.0348006 1.0348003 1.0347999
 1.0347996 1.0347993 1.034799  1.0347989 1.0347986 1.0347983 1.0347983
 1.034798  1.0347979 1.0347977 1.0347974 1.0347973 1.0347971 1.0347971
 1.0347968 1.0347968 1.0347966 1.0347966 1.0347967 1.0347966 1.0347966
 1.0347965 1.0347965 1.0347965 1.0347965 1.0347965 1.0347965 1.0347967
 1.0347967 1.0347967 1.0347968 1.0347971 1.0347971 1.0347971 1.0347972
 1.0347974 1.0347977 1.0347978 1.034798  1.0347981 1.0347984 1.0347986
 1.0347989 1.034799  1.0347992 1.0347996 1.0347999 1.0348002 1.0348008
 1.034801  1.03480

In [ ]:
import matplotlib.pyplot as plt

rhat_mat = diag["r_hat"].reshape((10, 10))
plt.figure(figsize=(8, 10))
im = plt.imshow(rhat_mat, cmap="YlOrRd", vmin=0.0, vmax=1.1)
plt.colorbar(im, label="R-hat")
plt.title("Convergence heatmap (R-hat)")
plt.show()


In [ ]:
diag

In [ ]:

rhat_mat = diag["r_hat_folded"].reshape((10, 10))
plt.figure(figsize=(8, 10))
im = plt.imshow(rhat_mat, cmap="YlOrRd", vmin=0.0, vmax=1.1)
plt.colorbar(im, label="R-hat")
plt.title("Convergence heatmap (Folded R-hat)")
plt.show()


In [ ]:
from langevin_sampler import plot_transport_plans_traces

plot_transport_plans_traces(samples, 10, 10_000)